In [1]:
import pandas as pd

recommended = pd.read_csv("../data/raw/works_recommended.csv")
sanctioned = pd.read_csv("../data/raw/works_sanctioned.csv")
completed = pd.read_csv("../data/raw/works_completed.csv")
expenditure = pd.read_csv("../data/raw/expenditure.csv")
allocated = pd.read_csv("../data/raw/allocated_limit.csv")

In [2]:
print(recommended.columns)
print(sanctioned.columns)
print(completed.columns)
print(expenditure.columns)
print(allocated.columns)

Index(['Sr. No.', 'Work category', 'work', 'State', 'IDA',
       'Hon'ble Members of Parliament', 'Constituency', 'Work description',
       'Recommended date', 'RECOMMENDED AMOUNT ( ₹ )', 'Sanction Date'],
      dtype='object')
Index(['Sr. No.', 'Work category', 'work', 'State', 'IDA',
       'Hon'ble Members of Parliament', 'Constituency', 'Work description',
       'Recommended date', 'Sanction Date', 'Sanction Amount ( ₹ )',
       'Work Status'],
      dtype='object')
Index(['Sr. No.', 'Work category', 'work', 'State', 'IDA', 'Work Description',
       'Hon'ble Members of Parliament', 'Constituency', 'Image',
       'Completion Date', 'Amount Disbursed ( ₹ )'],
      dtype='object')
Index(['Sr. No.', 'State', 'work', 'work_id', 'IDA',
       'Hon'ble Members of Parliament', 'Constituency', 'Expenditure Date',
       'Vendor Name', 'Payment Status', 'Fund Disbursed Amount ( ₹ )'],
      dtype='object')
Index(['Sr. No.', 'State', 'Hon'ble Members of Parliaments', 'Constituency',
  

# Work recommended dataset

In [3]:
print("Shape:", recommended.shape)
print("\nDtypes:")
print(recommended.dtypes)

print("\nMissing Values:")
print(recommended.isnull().sum())

print("\nDuplicate Rows:")
print(recommended.duplicated().sum())

print("\nFirst 5 Rows:")
print(recommended.head())

print("\nLast 5 Rows:")
print(recommended.tail())

Shape: (73000, 11)

Dtypes:
Sr. No.                            int64
Work category                     object
work                              object
State                             object
IDA                               object
Hon'ble Members of Parliament     object
Constituency                      object
Work description                  object
Recommended date                  object
RECOMMENDED AMOUNT ( ₹ )         float64
Sanction Date                     object
dtype: object

Missing Values:
Sr. No.                             0
Work category                       0
work                                0
State                               0
IDA                                 0
Hon'ble Members of Parliament       0
Constituency                        0
Work description                   99
Recommended date                    0
RECOMMENDED AMOUNT ( ₹ )            0
Sanction Date                    6944
dtype: int64

Duplicate Rows:
0

First 5 Rows:
   Sr. No.      Work cate

In [4]:
for col in recommended.columns:
    print("="*80)
    print(col)
    print("Unique:", recommended[col].nunique(dropna=False))
    if recommended[col].nunique(dropna=False) <= 20:
        print(recommended[col].value_counts(dropna=False))

Sr. No.
Unique: 73000
Work category
Unique: 4
Work category
Normal/Others            71408
Repair and Renovation     1098
Trust and Society          484
Bar and Associations        10
Name: count, dtype: int64
work
Unique: 66151
State
Unique: 36
IDA
Unique: 749
Hon'ble Members of Parliament
Unique: 532
Constituency
Unique: 532
Work description
Unique: 66739
Recommended date
Unique: 609
RECOMMENDED AMOUNT ( ₹ )
Unique: 13353
Sanction Date
Unique: 649


In [5]:
text_cols = recommended.select_dtypes(include="object").columns
for col in text_cols:
      print(f"\n{col}")
      print("Leading/Trailing Spaces:",(recommended[col].astype(str) != recommended[col].astype(str).str.strip()).sum())
      print("Blank Strings:",recommended[col].astype(str).str.strip().eq("").sum())


Work category
Leading/Trailing Spaces: 0
Blank Strings: 0

work
Leading/Trailing Spaces: 0
Blank Strings: 0

State
Leading/Trailing Spaces: 0
Blank Strings: 0

IDA
Leading/Trailing Spaces: 0
Blank Strings: 0

Hon'ble Members of Parliament
Leading/Trailing Spaces: 0
Blank Strings: 0

Constituency
Leading/Trailing Spaces: 0
Blank Strings: 0

Work description
Leading/Trailing Spaces: 0
Blank Strings: 0

Recommended date
Leading/Trailing Spaces: 0
Blank Strings: 0

Sanction Date
Leading/Trailing Spaces: 0
Blank Strings: 0


#### cleaning work_id, work_name

In [6]:
recommended["work"].head(5).tolist()

['WS/MP620/2024-2025/133166-Construction of buildings for community cultural activities',
 'WS/MP620/2025-2026/133167-Construction of rooms and halls in school and colleges',
 'WS/MP620/2024-2025/133190-Construction of buildings for community cultural activities',
 'WS/MP620/2025-2026/133191-Construction of buildings for community cultural activities',
 'WS/MP620/2024-2025/133301-Construction of buildings for community cultural activities']

In [7]:
print(recommended["work"].str.startswith("WS/").sum())
print(recommended["work"].str.startswith("NA-").sum())

66056
6944


In [8]:
recommended["work_clean"] = (
    recommended["work"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
recommended["work_id"] = (
    recommended["work_clean"]
    .str.extract(r'^(WS/\s*MP\d+/\d{4}-\d{4}/\d+)')[0]
    .str.replace(r"\s+", "", regex=True)
)
recommended["work_name"] = (
    recommended["work_clean"]
    .str.replace(r'^WS/\s*MP\d+/\d{4}-\d{4}/\d+-','',regex=True)
    .str.replace(r'^NA-','',regex=True)
    .str.strip()
)

In [9]:
recommended.drop(columns=["work_clean"], inplace=True)

In [11]:
print("Rows:", len(recommended))
print("WS rows:", recommended["work"].str.contains("WS/", na=False).sum())
print("Extracted IDs:", recommended["work_id"].notna().sum())
print("Unique IDs:", recommended["work_id"].nunique())
print("Missing IDs:", recommended["work_id"].isna().sum())

Rows: 73000
WS rows: 66056
Extracted IDs: 66056
Unique IDs: 66056
Missing IDs: 6944


In [12]:
recommended.columns

Index(['Sr. No.', 'Work category', 'work', 'State', 'IDA',
       'Hon'ble Members of Parliament', 'Constituency', 'Work description',
       'Recommended date', 'RECOMMENDED AMOUNT ( ₹ )', 'Sanction Date',
       'work_id', 'work_name'],
      dtype='object')

#### observed missing sanction date and work_desc

In [13]:
na_works = recommended[recommended["work_id"].isna()]
print("Total NA works:", len(na_works))
print("NA works with sanction date:",na_works["Sanction Date"].notna().sum())
print("NA works without sanction date:",na_works["Sanction Date"].isna().sum())

Total NA works: 6944
NA works with sanction date: 0
NA works without sanction date: 6944


In [14]:
missing_desc = recommended[recommended["Work description"].isna()]
print("Missing descriptions:", len(missing_desc))
print("Missing description + NA work:",missing_desc["work_id"].isna().sum())
print("Missing description + WS work:",missing_desc["work_id"].notna().sum())

Missing descriptions: 99
Missing description + NA work: 7
Missing description + WS work: 92


In [15]:
missing_ws = recommended[(recommended["Work description"].isna()) & (recommended["work_id"].notna())]
print("Rows:", len(missing_ws))
print("Null sanction:",missing_ws["Sanction Date"].isna().sum())
print("Non-null sanction:",missing_ws["Sanction Date"].notna().sum())

Rows: 92
Null sanction: 0
Non-null sanction: 92


In [17]:
recommended = recommended.rename(columns={
    "Sr. No.": "sr_no",
    "Work category": "work_category",
    "work": "work_raw",
    "State": "state",
    "IDA": "ida",
    "Hon'ble Members of Parliament": "mp_name",
    "Constituency": "constituency",
    "Work description": "work_description",
    "Recommended date": "recommended_date",
    "RECOMMENDED AMOUNT ( ₹ )": "recommended_amount",
    "Sanction Date": "sanction_date"
})

In [18]:
recommended["recommended_date"] = pd.to_datetime(
    recommended["recommended_date"],
    format="%d-%b-%Y",
    errors="coerce"
)

recommended["sanction_date"] = pd.to_datetime(
    recommended["sanction_date"],
    format="%d-%b-%Y",
    errors="coerce"
)

recommended["has_work_id"] = (
    recommended["work_id"]
    .notna()
    .astype(int)
)

recommended["is_sanctioned"] = (
    recommended["sanction_date"]
    .notna()
    .astype(int)
)

recommended["missing_description"] = (
    recommended["work_description"]
    .isna()
    .astype(int)
)

recommended["days_to_sanction"] = (
    recommended["sanction_date"]
    - recommended["recommended_date"]
).dt.days

In [19]:
recommended.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73000 entries, 0 to 72999
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   sr_no                73000 non-null  int64         
 1   work_category        73000 non-null  object        
 2   work_raw             73000 non-null  object        
 3   state                73000 non-null  object        
 4   ida                  73000 non-null  object        
 5   mp_name              73000 non-null  object        
 6   constituency         73000 non-null  object        
 7   work_description     72901 non-null  object        
 8   recommended_date     73000 non-null  datetime64[ns]
 9   recommended_amount   73000 non-null  float64       
 10  sanction_date        66056 non-null  datetime64[ns]
 11  work_id              66056 non-null  object        
 12  work_name            73000 non-null  object        
 13  missing_description  73000 non-

In [77]:
recommended_clean = recommended.copy()

In [78]:
recommended.to_csv("../data/processed/works_recommended_processed.csv", index=False)

# work sanctioned data

In [20]:
print("Shape:", sanctioned.shape)

print("\nDtypes:")
print(sanctioned.dtypes)

print("\nMissing Values:")
print(sanctioned.isnull().sum())

print("\nDuplicate Rows:")
print(sanctioned.duplicated().sum())

print("\nUnique work values:")
print(sanctioned["work"].nunique())

print("\nSample work values:")
print(sanctioned["work"].head(10).tolist())

Shape: (77999, 12)

Dtypes:
Sr. No.                            int64
Work category                     object
work                              object
State                             object
IDA                               object
Hon'ble Members of Parliament     object
Constituency                      object
Work description                  object
Recommended date                  object
Sanction Date                     object
Sanction Amount ( ₹ )            float64
Work Status                       object
dtype: object

Missing Values:
Sr. No.                           0
Work category                     0
work                              0
State                             0
IDA                               0
Hon'ble Members of Parliament     0
Constituency                      0
Work description                 98
Recommended date                  0
Sanction Date                     0
Sanction Amount ( ₹ )             0
Work Status                       0
dtype: int64

Dup

In [21]:
sanctioned['Recommended date'] = pd.to_datetime(
    sanctioned["Recommended date"],
    format="%d-%b-%Y",
    errors="coerce"
)
sanctioned['Sanction Date'] = pd.to_datetime(
    sanctioned["Sanction Date"],
    format="%d-%b-%Y",
    errors="coerce"
)

In [22]:
sanctioned["work_clean"] = (
    sanctioned["work"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

sanctioned["work_id"] = (
    sanctioned["work_clean"]
    .str.extract(r'^(WS/\s*MP\d+/\d{4}-\d{4}/\d+)')[0]
    .str.replace(r"\s+", "", regex=True)
)

sanctioned["work_name"] = (
    sanctioned["work_clean"]
    .str.replace(r'^WS/\s*MP\d+/\d{4}-\d{4}/\d+-','',regex=True)
    .str.strip()
)

sanctioned.drop(columns=["work_clean"], inplace=True)

In [23]:
print("Rows:", len(sanctioned))
print("Unique work_id:", sanctioned["work_id"].nunique())
print("Missing work_id:", sanctioned["work_id"].isna().sum())
print("Duplicate valid work_id:",sanctioned.loc[sanctioned["work_id"].notna(),"work_id"].duplicated().sum())

Rows: 77999
Unique work_id: 77999
Missing work_id: 0
Duplicate valid work_id: 0


In [24]:
recommended_ids = set(recommended["work_id"].dropna())
sanctioned_ids = set(sanctioned["work_id"].dropna())
print("Recommended IDs:",len(recommended_ids))
print("Sanctioned IDs:",len(sanctioned_ids))
print("Common IDs:",len(recommended_ids & sanctioned_ids))
print("Sanctioned but not Recommended:",len(sanctioned_ids - recommended_ids))
print("Recommended but not Sanctioned:",len(recommended_ids - sanctioned_ids))

Recommended IDs: 66056
Sanctioned IDs: 77999
Common IDs: 66056
Sanctioned but not Recommended: 11943
Recommended but not Sanctioned: 0


In [25]:
extra_ids = (sanctioned_ids - recommended_ids)
sanctioned_only = sanctioned[sanctioned["work_id"].isin(extra_ids)]
print(len(sanctioned_only))
print(sanctioned_only[["work_id", "Recommended date", "Sanction Date"]].head(20))
print(sanctioned_only["Recommended date"].min())
print(sanctioned_only["Recommended date"].max())

11943
                          work_id Recommended date Sanction Date
407     WS/MP431/2024-2025/135540       2024-08-13    2025-01-30
789     WS/MP341/2025-2026/136725       2024-08-22    2026-02-09
2362  WS/MP18007/2024-2025/140066       2024-09-09    2024-10-09
2882    WS/MP313/2024-2025/140993       2024-09-15    2025-02-19
3046  WS/MP18152/2024-2025/141227       2024-09-16    2024-10-18
3329    WS/MP423/2024-2025/141872       2024-09-20    2025-03-01
4970  WS/MP18058/2024-2025/145165       2024-10-02    2024-12-13
5112    WS/MP384/2024-2025/145501       2024-10-04    2025-02-07
6184    WS/MP384/2024-2025/147618       2024-10-16    2025-01-20
6385  WS/MP18076/2024-2025/148068       2024-10-17    2025-03-17
7051    WS/MP582/2026-2027/149091       2024-10-23    2026-08-25
7055    WS/MP431/2024-2025/149099       2024-10-23    2025-02-21
7084    WS/MP431/2026-2027/149155       2024-10-23    2026-05-21
7183    WS/MP478/2024-2025/149447       2024-10-25    2024-11-27
7184    WS/MP478/20

In [26]:
print(
    sanctioned["Work Status"]
    .value_counts(dropna=False)
)

Work Status
Physical Inspection         33427
Sanction                    20135
Vendor Identification       11404
Work partially Completed     7855
Work Completed               4371
Time Estimation               807
Name: count, dtype: int64


# work completed dataset

In [27]:
print("Rows:", len(completed))
print("Unique work:", completed["work"].nunique())
print("Duplicate rows:", completed.duplicated().sum())
print(completed.isnull().sum())

Rows: 33856
Unique work: 33856
Duplicate rows: 0
Sr. No.                             0
Work category                       0
work                                0
State                               0
IDA                                 0
Work Description                   79
Hon'ble Members of Parliament       0
Constituency                        0
Image                            9303
Completion Date                     0
Amount Disbursed ( ₹ )             96
dtype: int64


In [28]:
completed["work_clean"] = (
    completed["work"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

completed["work_id"] = (
    completed["work_clean"]
    .str.extract(r'^(WS/\s*MP\d+/\d{4}-\d{4}/\d+)')[0]
    .str.replace(r"\s+", "", regex=True)
)

completed["work_name"] = (
    completed["work_clean"]
    .str.replace(r'^WS/\s*MP\d+/\d{4}-\d{4}/\d+-','',regex=True)
    .str.strip()
)
completed.drop(columns=["work_clean"], inplace=True)

In [29]:
print("Unique work_id:", completed["work_id"].nunique())
print("Missing work_id:", completed["work_id"].isna().sum())

print(
    "Duplicate valid work_id:",
    completed.loc[
        completed["work_id"].notna(),
        "work_id"
    ].duplicated().sum()
)

Unique work_id: 33856
Missing work_id: 0
Duplicate valid work_id: 0


In [31]:
completed_ids = set(
    completed["work_id"].dropna()
)

print(
    "Completed IDs:",
    len(completed_ids)
)

print(
    "Completed ∩ Sanctioned:",
    len(completed_ids & sanctioned_ids)
)

print(
    "Completed not in Sanctioned:",
    len(completed_ids - sanctioned_ids)
)

Completed IDs: 33856
Completed ∩ Sanctioned: 33856
Completed not in Sanctioned: 0


# expenditure dataset

In [32]:
def clean_work_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\s+", "", regex=True)
    )

In [33]:
expenditure["work_id"] = clean_work_id(
    expenditure["work_id"]
)

In [34]:
expenditure["Expenditure Date"] = pd.to_datetime(
    expenditure["Expenditure Date"],
    format="%d-%b-%Y",
    errors="coerce"
)

In [35]:
exp_agg = (
    expenditure
    .groupby("work_id")
    .agg(
        total_disbursed=(
            "Fund Disbursed Amount ( ₹ )",
            "sum"
        ),
        avg_disbursement=(
            "Fund Disbursed Amount ( ₹ )",
            "mean"
        ),
        payment_count=(
            "work_id",
            "size"
        ),
        successful_payments=(
            "Payment Status",
            lambda x: (x == "Payment Success").sum()
        ),
        inprogress_payments=(
            "Payment Status",
            lambda x: (x == "Payment In-Progress").sum()
        ),
        first_payment_date=(
            "Expenditure Date",
            "min"
        ),
        latest_payment_date=(
            "Expenditure Date",
            "max"
        )
    )
    .reset_index()
)

In [37]:
print(exp_agg['successful_payments'].value_counts())
print(exp_agg['inprogress_payments'].value_counts())

successful_payments
1     5160
0     2348
2      360
3       73
4       63
5       42
6       25
8       14
10      14
11      13
9       13
7       11
13       9
12       6
16       5
15       5
14       5
18       3
20       3
21       3
22       2
24       1
23       1
17       1
19       1
Name: count, dtype: int64
inprogress_payments
0     5747
1     2303
2       64
3       15
6        9
5        6
4        6
7        5
8        4
13       3
11       3
10       3
29       2
21       2
12       2
14       2
15       2
18       1
9        1
20       1
Name: count, dtype: int64


In [36]:
print(expenditure["work_id"].duplicated().sum())

2819


In [38]:
print("Rows:", len(expenditure))
print("Unique work_id:", expenditure["work_id"].nunique() if "work_id" in expenditure.columns else "not created")
print(expenditure.isnull().sum())
print(expenditure.duplicated().sum())
print(expenditure["Payment Status"].value_counts(dropna=False))

Rows: 11000
Unique work_id: 8181
Sr. No.                          0
State                            0
work                             0
work_id                          0
IDA                              0
Hon'ble Members of Parliament    0
Constituency                     0
Expenditure Date                 0
Vendor Name                      0
Payment Status                   0
Fund Disbursed Amount ( ₹ )      0
dtype: int64
0
Payment Status
Payment Success        8018
Payment In-Progress    2982
Name: count, dtype: int64


In [39]:
print("Unique work_id:", expenditure["work_id"].nunique())
print("Missing work_id:", expenditure["work_id"].isna().sum())
print("Duplicate valid work_id:", expenditure.loc[expenditure["work_id"].notna(),"work_id"].duplicated().sum())

Unique work_id: 8181
Missing work_id: 0
Duplicate valid work_id: 2819


In [40]:
expenditure_ids = set(expenditure["work_id"].dropna())
print("Expenditure IDs:",len(expenditure_ids))
print("Expenditure ∩ Sanctioned:",len(expenditure_ids & sanctioned_ids))
print("Expenditure not in Sanctioned:",len(expenditure_ids - sanctioned_ids))

Expenditure IDs: 8181
Expenditure ∩ Sanctioned: 8181
Expenditure not in Sanctioned: 0


In [41]:
exp_counts = (expenditure["work_id"].value_counts())
print(exp_counts.describe())
print(exp_counts[exp_counts > 1].head(5))

count    8181.000000
mean        1.344579
std         1.677174
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        30.000000
Name: count, dtype: float64
work_id
WS/MP719/2025-2026/230890      30
WS/MP719/2025-2026/208008      29
WS/MP478/2025-2026/243395      24
WS/MP719/2025-2026/230888      23
WS/MP18162/2025-2026/251136    22
Name: count, dtype: int64
